## tl;dr

This notebook audits completion of the PAML M0 run and compares valid PAML omega values with the prior KaKs_Calculator NG values. It is a data-analysis artifact only; it does not modify genome, alignment, tree, or workflow inputs.

## Context & Methods

PAML input: `MS_REVISION/DNDS_ALT_MODELS/results_all_paml_m0/`. Prior Ka/Ks input: `Results/DNDS/RSSC_kaks2_df.csv`. The comparison joins exact gene-family identifiers and classifies positive selection as omega or Ka/Ks greater than 1. PAML M0 is a phylogeny-aware one-ratio estimate; the prior KaKs_Calculator value is the NG result retained from each family TSV.

### Key Assumptions

- Only PAML records with completed status and a valid one-row summary are used.
- The exact family identifier is the join key.
- Values at PAML boundary estimates (0.0001 or 999) are retained but flagged for interpretation.
- The two estimators are compared descriptively and are not assumed interchangeable.

## Data

The analysis script scans the PAML status ledger and individual summary files, validates numeric fields, joins to the prior Ka/Ks table, and writes auditable tabular outputs under `MS_REVISION/DNDS_PAML_VS_KAKS/`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / 'MS_REVISION').exists():
    repo = repo.parent
analysis_dir = repo / 'MS_REVISION/DNDS_PAML_VS_KAKS'
metrics = pd.read_csv(analysis_dir / 'comparison_metrics.tsv', sep='\t')
matched = pd.read_csv(analysis_dir / 'paml_vs_kaks_matched.tsv', sep='\t')
completion = pd.read_csv(analysis_dir / 'paml_completion_audit.tsv', sep='\t')
metrics.head()

## Results

The executed comparison reports 9,187 valid PAML summaries out of 9,624 eligible families and 6,271 exact family matches with the prior Ka/Ks table. PAML calls 1.93% of matched families positive versus 21.9% for the prior method.

In [ ]:
metrics.set_index('metric').loc[[
    'paml_eligible_gene_families', 'paml_usable_summaries',
    'paml_prepare_failed', 'paml_not_run', 'matched_gene_families',
    'paml_positive_fraction', 'old_kaks_positive_fraction',
    'spearman_raw_rho', 'selection_class_agreement_fraction',
    'selection_cohen_kappa'
]]

In [ ]:
ax = matched.plot.scatter(x='log10_kaks', y='log10_paml', alpha=0.25, s=8, figsize=(7, 6))
lo = min(matched[['log10_kaks', 'log10_paml']].min())
hi = max(matched[['log10_kaks', 'log10_paml']].max())
ax.plot([lo, hi], [lo, hi], '--', color='black', linewidth=1)
ax.set_xlabel('KaKs_Calculator NG: log10(Ka/Ks + 1e-6)')
ax.set_ylabel('PAML M0: log10(omega + 1e-6)')
ax.set_title('PAML M0 versus prior Ka/Ks')
plt.show()

## Takeaways

The methods show moderate rank ordering but materially different positive-selection rates. The prior method produces 1,287 Ka/Ks-positive / PAML-purifying disagreements versus 34 PAML-positive / prior-purifying disagreements. Boundary values and the incomplete PAML universe should be reviewed before gene-specific claims.

Notebook execution note: the current environment has `nbformat` but no installed Jupyter kernel/`nbclient`; execute this notebook in a Python Jupyter environment after the derived TSV outputs have been generated by `scripts/compare_paml_kaks.py`.